In [1]:
# Imports i wczytanie danych

import pandas as pd
import numpy as np
from datetime import datetime
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
# Jeśli uruchamiasz znotebooks/, przejdź poziom wyżej
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Dodatkowe zabezpieczenie: idź w górę, aż znajdziesz run_etl.py
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "run_etl.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

# Ścieżki względem root
RAW_PATH = PROJECT_ROOT / "data/raw/online_retail_II.xlsx"
DOCS_DIR = PROJECT_ROOT / "docs"
INTERIM_DIR = PROJECT_ROOT / "data/interim"
DOCS_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DIR.mkdir(parents=True, exist_ok=True)


from run_etl import load_raw_excel, clean_raw

raw = load_raw_excel(RAW_PATH)
print(f"RAW: {len(raw):,} wierszy")

[2025-09-02 15:22:17] Ładowanie surowych danych z: C:\Users\pzoladkiewicz\Documents\senior-bi-craftsman-journey\projects\Retail-Omnichannel-Optimization\data\raw\online_retail_II.xlsx
[2025-09-02 15:23:27] Wczytano 1,067,371 rekordów z 2 arkuszy
RAW: 1,067,371 wierszy


In [2]:
# Definicja clean_raw_v2

def clean_raw_v2(df_raw: pd.DataFrame) -> pd.DataFrame:
    req = ["Invoice", "StockCode", "Description", "Quantity", "InvoiceDate", "Price", "Customer ID", "Country"]
    missing = [c for c in req if c not in df_raw.columns]
    if missing:
        raise ValueError(f"Brak kolumn: {missing}")

    df = df_raw.copy()

    # Typy danych
    df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
    df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
    df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")

    # Normalizacja stringów
    for c in ["Invoice", "StockCode", "Description", "Country"]:
        df[c] = df[c].astype(str).str.strip()

    df["Customer ID"] = (
        df["Customer ID"].astype (str).str.strip()
        .replace({"nan": np.nan, "": np.nan})
    )

    # Deduplikacja
    before = len(df)
    df = df.drop_duplicates(subset=["Invoice", "StockCode", "Quantity", "InvoiceDate", "Price"], keep="first")

    # KLUCZOWA ZMIANA: Filtruj tylko oczywiste błędy (BEZ Quantity>0 i Price>0)
    mask = (
        df["InvoiceDate"].notna() &
        (df["InvoiceDate"] <= pd.Timestamp.now()) &
        df["Invoice"].ne("") &
        df["StockCode"].ne("")
    )
    df = df.loc[mask].copy()

    # Normalizacja opisów
    df["Description"] = df["Description"].replace({"nan": "", "None": ""}).fillna("").str.strip()

    # Total Value (może być ujemne!)
    df["Total_Value"] = df["Quantity"].astype(float) * df["Price"].astype(float)

    return df

clean_v2 = clean_raw_v2(raw)
clean_v1 = clean_raw(raw)

print(f"CLEAN_v1 (stara): {len(clean_v1):,}")
print(f"CLEAN_v2 (nowa): {len(clean_v2):,}")
print(f"Różnica: +{len(clean_v2) - len(clean_v1):,} rekordów")
    

[2025-09-02 15:23:30] Walidacja obecnosci kolumn RAW...
[2025-09-02 15:23:30] Konwersja typów (Quantity, Price, InvoiceDate)...
[2025-09-02 15:23:32] Deduplikacja rekordów...
[2025-09-02 15:23:33] Usunięto duplikatów: 34,337
[2025-09-02 15:23:33] Filtrowanie prawidłowych sprzedaży...
[2025-09-02 15:23:33] Po filtrowaniu: 1,007,912 rekordów
[2025-09-02 15:23:33] Standaryzacja krajów...
[2025-09-02 15:23:33] Czyszczenie opisów produktu...
[2025-09-02 15:23:33] Kalkulacja Total_Value...
[2025-09-02 15:23:33] Łączny przychód: £20,476,082.17
[2025-09-02 15:23:33] Quality Gates po czyszczeniu...
CLEAN_v1 (stara): 1,007,912
CLEAN_v2 (nowa): 1,033,034
Różnica: +25,122 rekordów


In [3]:
# Szczegółowa analiza różnic

def summarize(df, label):
    q_neg = (df["Quantity"] < 0).sum()
    p_neg = (df["Price"] < 0).sum()
    p_zero = (df["Price"] == 0).sum()
    inv_C = df["Invoice"].str.startswith("C", na=False).sum()
    inv_A = df["Invoice"].str.startswith("A", na=False).sum()
    total_value = df["Total_Value"].sum()

    print(f"[{label}]")
    print(f" Rekordy: {len(df):,}")
    print(f" Quantity < 0: {q_neg:,}")
    print(f" Price < 0: {p_neg:,}")
    print(f" Price = 0: {p_zero:,}")
    print(f" Faktury C*: {inv_C:,}")
    print(f" Faktury A*: {inv_A:,}")
    print(f" Total Value: {total_value:,}")
    print()

summarize(clean_v1, "CLEAN v1 (obecna)")
summarize(clean_v2, "CLEAN v2 (poprawiona)")

# Przykłady utraconych danych
lost_idx = clean_v2.index.difference(clean_v1.index)
lost_data = clean_v2.loc[lost_idx]

print(f"UTRACONE w v1: {len(lost_data):,} rekordów")

if len(lost_data) > 0:
    
    print("\nPrzykłady zwrotów (Quantity < 0):")
    returns = lost_data[lost_data["Quantity"] < 0].head(5)
    if len(returns) > 0:
        print(returns[["Invoice", "StockCode", "Description", "Quantity", "Price", "Total_Value"]].to_string(index=False))

    print("\nPrzykłady korekt (Price < 0):")
    adjustments = lost_data[lost_data["Price"] < 0].head(5)
    if len(adjustments) > 0:
        print(adjustments[["Invoice", "StockCode", "Description", "Quantity", "Price", "Total_Value"]].to_string(index=False))
        
    

[CLEAN v1 (obecna)]
 Rekordy: 1,007,912
 Quantity < 0: 0
 Price < 0: 0
 Price = 0: 0
 Faktury C*: 1
 Faktury A*: 1
 Total Value: 20,476,082.167999998

[CLEAN v2 (poprawiona)]
 Rekordy: 1,033,034
 Quantity < 0: 22,496
 Price < 0: 5
 Price = 0: 6,014
 Faktury C*: 19,104
 Faktury A*: 6
 Total Value: 18,854,981.847999997

UTRACONE w v1: 25,122 rekordów

Przykłady zwrotów (Quantity < 0):
Invoice StockCode                   Description  Quantity  Price  Total_Value
C489449     22087      PAPER BUNTING WHITE LACE       -12   2.95        -35.4
C489449    85206A  CREAM FELT EASTER EGG BASKET        -6   1.65         -9.9
C489449     21895 POTTING SHED SOW 'N' GROW SET        -4   4.25        -17.0
C489449     21896            POTTING SHED TWINE        -6   2.10        -12.6
C489449     22083    PAPER CHAIN KIT RETRO SPOT       -12   2.95        -35.4

Przykłady korekt (Price < 0):
Invoice StockCode     Description  Quantity     Price  Total_Value
A506401         B Adjust bad debt         1 -535

In [5]:
# Zapisz poprawione dane do interim
clean_v2.to_csv(INTERIM_DIR / "clean_v2_for_classification.csv", index=False, encoding="utf-8")
clean_v1.to_csv(INTERIM_DIR / "clean_v1_for_classification.csv", index=False, encoding="utf-8")

# Zapisz statystyki porównania
comparison_stats = pd.DataFrame({
    'Metryka': ['Total_Records', 'Qty_Negative', 'Price_Negative', 'Price_Zero', 'Invoice_C', 'Invoice_A', 'Total_Value_GBP'],
    'clean_v1': [
        len(clean_v1),
        (clean_v1['Quantity']<0).sum(),
        (clean_v1['Price']<0).sum(),
        (clean_v1['Price']==0).sum(),
        clean_v1['Invoice'].str.startswith('C', na=False).sum(),
        clean_v1['Invoice'].str.startswith('A', na=False).sum(),
        clean_v1['Total_Value'].sum()
    ],
    'clean_v2': [
        len(clean_v2),
        (clean_v2['Quantity']<0).sum(),
        (clean_v2['Price']<0).sum(),
        (clean_v2['Price']==0).sum(),
        clean_v2['Invoice'].str.startswith('C', na=False).sum(),
        clean_v2['Invoice'].str.startswith('A', na=False).sum(),
        clean_v2['Total_Value'].sum()
    ]
})

comparison_stats['Różnica'] = comparison_stats['clean_v2'] - comparison_stats['clean_v1']
print("\nSTATYSTYKI PORÓWNANIA:")
print(comparison_stats.to_string(index=False))

# Zapisz do pliku
comparison_stats.to_csv(DOCS_DIR / "clean_raw_comparison_stats.csv", index=False)

print(f"\nZapisano pliki:")
print(f"- data/interin/clean_v1_for_classification.csv")
print(f"- data/interin/clean_v2_for_classification.csv")
print(f"- docs/clean_raw_comparison_stats.csv")

print(f"\n{'=' * 50}")
print(f"REKOMENDACJA: {'PRZYJĄĆ clean_raw_v2' if len(clean_v2) > len(clean_v1) else 'ZOSTAĆ przy v1'}")
print(f"{'=' * 50}")


STATYSTYKI PORÓWNANIA:
        Metryka     clean_v1     clean_v2     Różnica
  Total_Records  1007912.000  1033034.000    25122.00
   Qty_Negative        0.000    22496.000    22496.00
 Price_Negative        0.000        5.000        5.00
     Price_Zero        0.000     6014.000     6014.00
      Invoice_C        1.000    19104.000    19103.00
      Invoice_A        1.000        6.000        5.00
Total_Value_GBP 20476082.168 18854981.848 -1621100.32

Zapisano pliki:
- data/interin/clean_v1_for_classification.csv
- data/interin/clean_v2_for_classification.csv
- docs/clean_raw_comparison_stats.csv

REKOMENDACJA: PRZYJĄĆ clean_raw_v2
